# LeetCode #684: Redundant Connection

https://leetcode.com/problems/redundant-connection/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **DFS/BFS per Edge** | $O(n^2)$ | $O(n)$ |
| **Optimal: Union-Find ★** | $O(n \cdot \alpha(n))$ | $O(n)$ |

---

## Understanding the Methods

### DFS/BFS per Edge
For each edge, check if the two endpoints are already connected using DFS/BFS on the graph built so far. The first edge that creates a cycle is the answer. Each connectivity check is O(n), giving O(n^2) total.

### Optimal: Union-Find ★
Process edges in order. For each edge (u, v), attempt to union them. If u and v are already in the same component (find(u) == find(v)), this edge forms a cycle and is the answer. Union-Find with path compression and union by rank gives near-constant time per operation.

**Why this is better than DFS/BFS:** Each union/find operation is effectively O(1) amortized (inverse Ackermann), making the total time nearly O(n).

**Constraints:**
* n == edges.length
* 3 <= n <= 1000
* edges[i].length == 2
* 1 <= ai < bi <= n
* There are no self-loops or repeated edges

## Solutions

### C#

In [ ]:
public class Solution {
    private int[] parent, rank;

    public int[] FindRedundantConnection(int[][] edges) {
        int n = edges.Length;
        parent = new int[n + 1];
        rank = new int[n + 1];
        for (int i = 0; i <= n; i++) parent[i] = i;
        foreach (var e in edges)
            if (!Union(e[0], e[1]))
                return e;
        return new int[0];
    }

    private int Find(int x) {
        if (parent[x] != x) parent[x] = Find(parent[x]);
        return parent[x];
    }

    private bool Union(int x, int y) {
        int px = Find(x), py = Find(y);
        if (px == py) return false;
        if (rank[px] < rank[py]) parent[px] = py;
        else if (rank[px] > rank[py]) parent[py] = px;
        else { parent[py] = px; rank[px]++; }
        return true;
    }
}

### Python

In [ ]:
class Solution:
    def findRedundantConnection(self, edges: list[list[int]]) -> list[int]:
        parent = list(range(len(edges) + 1))
        rank = [0] * (len(edges) + 1)

        def find(x):
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x, y):
            px, py = find(x), find(y)
            if px == py:
                return False
            if rank[px] < rank[py]:
                px, py = py, px
            parent[py] = px
            if rank[px] == rank[py]:
                rank[px] += 1
            return True

        for u, v in edges:
            if not union(u, v):
                return [u, v]
        return []

### Go

In [ ]:
func findRedundantConnection(edges [][]int) []int {
    n := len(edges)
    parent := make([]int, n+1)
    rank := make([]int, n+1)
    for i := range parent { parent[i] = i }

    var find func(int) int
    find = func(x int) int {
        if parent[x] != x { parent[x] = find(parent[x]) }
        return parent[x]
    }

    union := func(x, y int) bool {
        px, py := find(x), find(y)
        if px == py { return false }
        if rank[px] < rank[py] { px, py = py, px }
        parent[py] = px
        if rank[px] == rank[py] { rank[px]++ }
        return true
    }

    for _, e := range edges {
        if !union(e[0], e[1]) {
            return e
        }
    }
    return nil
}

### Rust

In [ ]:
impl Solution {
    pub fn find_redundant_connection(edges: Vec<Vec<i32>>) -> Vec<i32> {
        let n = edges.len();
        let mut parent: Vec<usize> = (0..=n).collect();
        let mut rank = vec![0usize; n + 1];

        fn find(parent: &mut Vec<usize>, x: usize) -> usize {
            if parent[x] != x { parent[x] = find(parent, parent[x]); }
            parent[x]
        }

        for e in &edges {
            let (u, v) = (e[0] as usize, e[1] as usize);
            let (pu, pv) = (find(&mut parent, u), find(&mut parent, v));
            if pu == pv { return e.clone(); }
            if rank[pu] < rank[pv] { parent[pu] = pv; }
            else if rank[pu] > rank[pv] { parent[pv] = pu; }
            else { parent[pv] = pu; rank[pu] += 1; }
        }
        vec![]
    }
}

## Example Scenarios

### Scenario 1: Simple triangle
**Input:** `edges = [[1,2],[1,3],[2,3]]`  
Adding [2,3] creates a cycle (1-2-3-1). **Output:** `[2,3]`

### Scenario 2: Last edge creates cycle
**Input:** `edges = [[1,2],[2,3],[3,4],[1,4],[1,5]]`  
Adding [1,4] creates a cycle (1-2-3-4-1). **Output:** `[1,4]`

### Scenario 3: Star with extra edge
**Input:** `edges = [[1,2],[1,3],[1,4],[2,3]]`  
1-2, 1-3 are connected, then [2,3] creates a cycle. **Output:** `[2,3]`

### Scenario 4: Chain with closing edge
**Input:** `edges = [[1,2],[2,3],[3,1]]`  
Linear chain 1-2-3, then [3,1] closes it. **Output:** `[3,1]`

### Scenario 5: Larger graph
**Input:** `edges = [[1,2],[2,3],[3,4],[4,5],[5,6],[6,3]]`  
Edge [6,3] creates a cycle 3-4-5-6-3. **Output:** `[6,3]`

*Infographic will be added in a future update.*